### Import libraries
Spark functions used in the checks below.

In [0]:
from pyspark.sql import functions as F

### Row counts after pipeline run
Confirms the gold tables match the expected counts. Re-running with no new data should change nothing.

In [0]:
for t, expected in [("dim_customers", 54), ("dim_products", 415), ("dim_gross_price", 811), ("fact_orders", 101212)]:
    actual = spark.table(f"fmcg.gold.{t}").count()
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{t}: {actual} (expected {expected}) → {status}")

dim_customers: 54 (expected 54) → OK
dim_products: 415 (expected 415) → OK
dim_gross_price: 811 (expected 811) → OK
fact_orders: 101212 (expected 101212) → OK


### December units and rows in fact_orders
Total December units and rows for both companies. Should stay the same when the pipeline runs without new files.

In [0]:
dec = (
    spark.table("fmcg.gold.fact_orders")
    .filter("date = '2025-12-01'")
    .agg(F.sum("sold_quantity").alias("units"), F.count("*").alias("rows"))
    .first()
)

print("December units:", dec["units"], "→", "OK" if dec["units"] == 5225651 else "CHANGED")
print("December rows: ", dec["rows"], "→", "OK" if dec["rows"] == 5097 else "CHANGED")

December units: 5225651 → OK
December rows:  5097 → OK


### December units by company
Splits December's orders in `fact_orders` into Atlikon and SportsBar using the customer's platform.

In [0]:
dec_by_company = (
    spark.table("fmcg.gold.fact_orders")
    .filter("date = '2025-12-01'")
    .join(spark.table("fmcg.gold.dim_customers").select("customer_code", "platform"), "customer_code")
    .withColumn("company", F.when(F.col("platform") == "Sports Bar", "SportsBar").otherwise("Atlikon"))
    .groupBy("company")
    .agg(F.sum("sold_quantity").alias("units"), F.count("*").alias("rows"))
)
display(dec_by_company)

company,units,rows
Atlikon,3398439,4485
SportsBar,1827212,612


### Date range by company
First month, last month and number of months for each company in `fact_orders`.

In [0]:
display(
    spark.table("fmcg.gold.fact_orders")
    .join(spark.table("fmcg.gold.dim_customers").select("customer_code", "platform"), "customer_code")
    .withColumn("company", F.when(F.col("platform") == "Sports Bar", "SportsBar").otherwise("Atlikon"))
    .groupBy("company")
    .agg(F.min("date").alias("first_month"), F.max("date").alias("last_month"), F.countDistinct("date").alias("months"))
)

company,first_month,last_month,months
SportsBar,2025-07-01,2025-12-01,6
Atlikon,2024-01-01,2025-12-01,24
